# AI Essay Detector - Model Comparison & Visualization

This notebook aggregates metrics across all four models, generates summary markdown comparison charts, and saves comparison figures.


### Kaggle Dataset Presence Check & Downloader
This block checks if the dataset `train_v4_drcat_01.csv` is present locally. If not, it sets up Kaggle, downloads the dataset zip, extracts it, and performs cleanup.

In [ ]:
import os

DATASET_FILE_NAME = "train_v4_drcat_01.csv"
KAGGLE_ZIP_FILE_NAME = "daigt-v4-train-dataset.zip"

# Check if the dataset file already exists locally
if os.path.exists(DATASET_FILE_NAME):
    print(f"'{DATASET_FILE_NAME}' found locally. Skipping Kaggle download.")
else:
    print(f"'{DATASET_FILE_NAME}' not found locally. Attempting to download from Kaggle...")

    # 1. Install the Kaggle library
    !pip install kaggle --quiet

    # Ensure the .kaggle directory exists (redundant if previous cell ran, but safe)
    !mkdir -p ~/.kaggle

    # The kaggle.json file should have been created by the previous cell
    # To prevent 'kaggle.json' from being publicly readable
    !chmod 600 ~/.kaggle/kaggle.json

    print("Kaggle API setup should be complete (kaggle.json created by previous cell).")

    # 3. Download the dataset
    dataset_name = "thedrcat/daigt-v4-train-dataset"
    !kaggle datasets download -d {dataset_name}

    # 4. Unzip the downloaded file
    !unzip -o {KAGGLE_ZIP_FILE_NAME} -d .

    # 5. Clean up the zip file (optional)
    !rm {KAGGLE_ZIP_FILE_NAME}

    if os.path.exists(DATASET_FILE_NAME):
        print(f"\nDataset '{dataset_name}' downloaded and unzipped successfully.")
        print(f"You should now see '{DATASET_FILE_NAME}' in your current directory.")
    else:
        raise FileNotFoundError(f"Failed to download and extract '{DATASET_FILE_NAME}' from Kaggle.")


### Working Directory & Path Initialization
Since this notebook runs inside the `notebooks/` directory, we shift the kernel's active path to the project root (`..`) so that relative data paths map correctly. We also add `src/` to the python import path so that custom helper modules like `evaluate` can be imported.

In [ ]:
import os
import sys

# If running from notebooks/ directory, switch to the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Ensure the src/ directory is in the import path so evaluate can be found
sys.path.append(os.path.abspath('src'))

print(f"Current Working Directory set to: {os.getcwd()}")
print("Import path appended. You can now import evaluation utilities successfully.")


### 1. Import Visualization & Comparison Libraries
We import standard scientific libraries and plotting styles (`seaborn`, `matplotlib`).


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


### 2. Loading & Plotting Logic
This block parses individual model JSON summary files from `results/` and aggregates them into comparative figures (discrimination ROC curves, Recall under strict conditions, and FPR reductions).


In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

def load_results():
    """Load cross-fold summaries from JSON files."""
    models_data = {}
    
    file_mapping = {
        "Logistic Regression": "logistic_cross_fold_summary.json",
        "Support Vector Machine": "svm_cross_fold_summary.json",
        "Multinomial Naive Bayes": "mnb_cross_fold_summary.json",
        "Random Forest": "rf_cross_fold_summary.json"
    }
    
    for name, filename in file_mapping.items():
        path = os.path.join(RESULTS_DIR, filename)
        if os.path.exists(path):
            with open(path, "r") as f:
                models_data[name] = json.load(f)
        else:
            print(f"Warning: Summary file {filename} not found at {path}.")
            
    return models_data

def compile_comparison_dataframe(models_data):
    """Transform the loaded JSON results into a clean pandas DataFrame for comparisons."""
    records = []
    metrics_keys = ["accuracy", "f1_score", "auc_roc", "precision", "recall", "specificity", "false_positive_rate"]
    
    for model_name, data in models_data.items():
        # Robust schema parsing (list of folds vs dict wrapper)
        if isinstance(data, list):
            fold_results = data
            thresholds = [f["best_threshold"] for f in fold_results if "best_threshold" in f]
            avg_threshold = np.mean(thresholds) if thresholds else 0.5
        elif isinstance(data, dict):
            fold_results = data["fold_results"]
            avg_threshold = data.get("global_tuned_threshold", 0.5)
        else:
            print(f"Error: Format of {model_name} results is unrecognized.")
            continue
            
        # Calculate mean metrics across all folds
        def_metrics = {}
        for key in metrics_keys:
            vals = [f["metrics_default"][key] for f in fold_results if key in f["metrics_default"]]
            def_metrics[key] = np.mean(vals) if vals else 0.0
            
        tuned_metrics = {}
        for key in metrics_keys:
            vals = [f["metrics_tuned"][key] for f in fold_results if key in f["metrics_tuned"]]
            tuned_metrics[key] = np.mean(vals) if vals else 0.0
            
        # 1. Default Threshold (0.5)
        rec_default = {
            "Model": model_name,
            "Setting": "Default (0.50)",
            "Accuracy": def_metrics["accuracy"] * 100,
            "F1-Score": def_metrics["f1_score"],
            "ROC-AUC": def_metrics["auc_roc"],
            "Precision": def_metrics["precision"] * 100,
            "Recall (Detection)": def_metrics["recall"] * 100,
            "Specificity": def_metrics["specificity"] * 100,
            "FPR": def_metrics["false_positive_rate"] * 100
        }
        records.append(rec_default)
        
        # 2. Tuned Safety Threshold
        rec_tuned = {
            "Model": model_name,
            "Setting": f"Tuned ({avg_threshold:.3f})",
            "Accuracy": tuned_metrics["accuracy"] * 100,
            "F1-Score": tuned_metrics["f1_score"],
            "ROC-AUC": tuned_metrics["auc_roc"],
            "Precision": tuned_metrics["precision"] * 100,
            "Recall (Detection)": tuned_metrics["recall"] * 100,
            "Specificity": tuned_metrics["specificity"] * 100,
            "FPR": tuned_metrics["false_positive_rate"] * 100
        }
        records.append(rec_tuned)
        
    return pd.DataFrame(records)

def plot_roc_auc_comparison(df_comparison):
    """Generate and save a bar chart comparing the ROC-AUC of all models."""
    plt.figure(figsize=(8, 5))
    
    # Filter default settings (AUC is the same for default/tuned, so we just pick one)
    df_auc = df_comparison[df_comparison["Setting"].str.contains("Default")].copy()
    df_auc = df_auc.sort_values(by="ROC-AUC", ascending=False)
    
    colors = sns.color_palette("viridis", len(df_auc))
    bars = plt.bar(df_auc["Model"], df_auc["ROC-AUC"], color=colors, edgecolor='grey', width=0.5)
    
    # Highlight value labels on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')
        
    plt.ylim(0.90, 1.01)  # Focus on the high-performance range
    plt.ylabel("ROC-AUC Score")
    plt.title("Classifier Discrimination Capability (ROC-AUC)")
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGE_DIR, "model_comparison_roc_auc.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved ROC-AUC comparison plot -> {plot_path}")

def plot_safety_recall_comparison(df_comparison):
    """Generate and save a chart comparing the Recall (AI detection) under tuned safety threshold."""
    plt.figure(figsize=(9, 5.5))
    
    # Filter tuned safety settings
    df_tuned = df_comparison[df_comparison["Setting"].str.contains("Tuned")].copy()
    df_tuned = df_tuned.sort_values(by="Recall (Detection)", ascending=False)
    
    colors = sns.color_palette("magma", len(df_tuned))
    bars = plt.bar(df_tuned["Model"], df_tuned["Recall (Detection)"], color=colors, edgecolor='grey', width=0.5)
    
    # Add values on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
        
    plt.ylim(0, 110)
    plt.ylabel("Recall / AI Detection Rate (%)")
    plt.xlabel("Model")
    plt.title("Recall / Detection Rate Under Strict Academic Integrity safety (FPR <= 0.1%)")
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGE_DIR, "model_comparison_tuned_recall.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved Tuned Recall comparison plot -> {plot_path}")

def plot_fpr_default_vs_tuned(df_comparison):
    """Plot comparative FPR rates before and after safety threshold tuning."""
    plt.figure(figsize=(9, 5.5))
    
    # Restructure dataframe for grouped bar plotting
    df_plot = df_comparison.copy()
    df_plot["Setting Type"] = df_plot["Setting"].apply(lambda x: "Default (0.50)" if "Default" in x else "Tuned (FPR <= 0.1%)")
    
    ax = sns.barplot(
        data=df_plot,
        x="Model",
        y="FPR",
        hue="Setting Type",
        palette="muted"
    )
    
    # Add value annotations on top of the bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.3f}%",
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='center',
                        xytext=(0, 7),
                        textcoords='offset points',
                        fontweight='bold', fontsize=9)
            
    # Add red line for target FPR boundary
    plt.axhline(0.1, color='red', linestyle='--', linewidth=1.5, label="Safety Threshold Ceiling (0.1%)")
    
    plt.ylabel("False Positive Rate (%)")
    plt.title("False Positive Rate (False Accusations): Default vs Tuned Threshold")
    plt.legend(loc="upper right")
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGE_DIR, "model_comparison_fpr_reduction.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"Saved FPR Comparison plot -> {plot_path}")

def main():
    print("=== Loading Model Summaries ===")
    models_data = load_results()
    
    if not models_data:
        print("No model results available. Please run the training scripts first.")
        return
        
    df_comparison = compile_comparison_dataframe(models_data)
    
    print("\n=== GLOBAL MODEL COMPARISON TABLE (CROSS-FOLD MEANS) ===")
    cols_to_show = ["Model", "Setting", "Accuracy", "F1-Score", "ROC-AUC", "Precision", "Recall (Detection)", "FPR"]
    print(df_comparison[cols_to_show].to_string(index=False))
    
    print("\n=== Generating Comparative Visualization Plots ===")
    plot_roc_auc_comparison(df_comparison)
    plot_safety_recall_comparison(df_comparison)
    plot_fpr_default_vs_tuned(df_comparison)
    print("\nVisualizations saved successfully to the results/ folder.")


### 3. Execution & Synthesis Output
Triggers the loader, logs a comparison table, and generates comparison png images in the workspace.


In [ ]:
if __name__ == "__main__":
    models_data = load_results()
    if models_data:
        comparison_df = compile_comparison_dataframe(models_data)
        print_comparison_markdown(comparison_df)
        save_comparison_plots(models_data)
